# Chapitre 12 - Cours 2 : SVM à noyaux, perte charnière, chemin de régularisation, régression

Ce notebook couvre la section 12.3 du livre :

1. l'**astuce du noyau** : le dual ne dépend que de produits scalaires (éq. (12.19)–(12.24)) ;
2. les SVM non linéaires sur la simulation « mixture » (figure 12.3) ;
3. le SVM vu comme une méthode **perte + pénalité** : la **perte charnière** comparée aux
   autres pertes (figure 12.4, tableau 12.1) ;
4. l'interprétation par les espaces de fonctions engendrés par un noyau, et la version
   « régression logistique à noyau » (figure 12.5) ;
5. le choix de $C$ et du paramètre du noyau (figure 12.6), le **chemin de régularisation**
   (figure 12.7) ;
6. les SVM pour la **régression** (figure 12.8) et la régression ridge à noyau
   (éq. (12.45)–(12.49)).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch12_svm import (SVMClassifier, SVMRegressor, polynomial_kernel, rbf_kernel,
                          poly2_features, hinge_loss, binomial_deviance, squared_error_loss,
                          huberized_hinge, eps_insensitive, huber_loss, kernel_ridge,
                          simulate_mixture, mixture_bayes_prob, plot_mixture_boundary)
set_style()

## 1. Agrandir l'espace des variables… sans le calculer

Pour obtenir des frontières non linéaires, on peut remplacer $x$ par un vecteur de
**fonctions de base** $h(x) = (h_1(x), \dots, h_M(x))$ (polynômes, splines… voir
chapitre 5) et appliquer le classifieur du cours 1 aux $h(x_i)$. Le dual (12.13) devient

$$L_D = \sum_i \alpha_i - \tfrac12\sum_i\sum_{i'}\alpha_i\alpha_{i'}y_iy_{i'}\,\langle h(x_i), h(x_{i'})\rangle \qquad (12.19)$$

et, grâce à $\beta = \sum_i \alpha_i y_i h(x_i)$, la fonction de décision s'écrit

$$f(x) = h(x)^T\beta + \beta_0 = \sum_i \alpha_i y_i\,\langle h(x), h(x_i)\rangle + \beta_0 \qquad (12.20).$$

**Observation clé.** Les deux formules ne font intervenir $h$ qu'à travers des produits
scalaires. Il suffit donc de connaître la fonction

$$K(x, x') = \langle h(x), h(x')\rangle \qquad (12.21),$$

appelée **noyau** : on n'a jamais besoin de calculer $h$ explicitement, même si $M$ est très
grand (voire infini).

**Quelles fonctions $K$ conviennent ?** $K$ doit être symétrique et **semi-définie
positive** : pour tous points $x_1,\dots,x_N$, la matrice de Gram $\mathbf{K} = [K(x_i, x_{i'})]$
vérifie $c^T\mathbf{K}c \ge 0$ pour tout $c\in\mathbb{R}^N$ (ses valeurs propres sont
positives). C'est évidemment nécessaire : $c^T\mathbf{K}c = \|\sum_i c_i h(x_i)\|^2$.

Trois noyaux classiques (éq. (12.22)) :

* polynomial de degré $d$ : $K(x,x') = (1 + \langle x, x'\rangle)^d$ ;
* radial (gaussien) : $K(x,x') = \exp(-\gamma\|x-x'\|^2)$ ;
* « réseau de neurones » : $K(x,x') = \tanh(\kappa_1\langle x,x'\rangle + \kappa_2)$.

**Exemple (12.23).** Pour $p = 2$ et $d = 2$, en développant le carré :

$$(1 + x_1x_1' + x_2x_2')^2 = 1 + 2x_1x_1' + 2x_2x_2' + (x_1x_1')^2 + (x_2x_2')^2 + 2x_1x_1'x_2x_2',$$

qui est bien $\langle h(x), h(x')\rangle$ avec
$h(x) = (1, \sqrt2x_1, \sqrt2x_2, x_1^2, x_2^2, \sqrt2x_1x_2)$, donc $M = 6$.

In [2]:
rng = np.random.RandomState(0)
A, B = rng.randn(5, 2), rng.randn(4, 2)
lhs = polynomial_kernel(A, B, degree=2)
rhs = poly2_features(A) @ poly2_features(B).T
print("écart max entre (1+<x,x'>)^2 et <h(x),h(x')> :", np.abs(lhs - rhs).max())

écart max entre (1+<x,x'>)^2 et <h(x),h(x')> : 3.552713678800501e-15


Vérifions numériquement la positivité : plus petite valeur propre de la matrice de Gram sur
200 points, pour chaque noyau. Le noyau tanh **n'est pas** semi-défini positif en général.